<a href="https://colab.research.google.com/github/Lordyblade/BigData26_B_2411532010_IbrahimMousaDhani/blob/main/Praktikum2/Copy_of_BD_B_P02_2411532010_IbrahimMousaDhani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Latihan 1** - Mengubah SEED menjadi **7** dan menjalankan ulang seluruh pipeline.

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.7 MB/s eta 0:00:00


### **K-1. Import Library dan Inisialisasi**

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA   = "/content/data"
DIR_SIMPAN  ="/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
['transaksi_mentah.csv', 'transaksi_bersih.csv']


### **K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)**

In [4]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()

    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
path_mentah = os.path.join(DIR_SIMPAN, "transaksi_mentah_latihan.csv")
df.to_csv(path_mentah, index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


In [5]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur Basic,Rumah Tangga,120000,2,Transfer Bank,14-09-2026,Bau-Bau,1.0
1,TRX00430,Faizah Kusumo,Voluptates Max,Olahraga,Rp250.000,3,Transfer Bank,2026-07-17,Cilegon,4.0
2,TRX00083,"Drs. Sari Aryani, M.TI.",Deserunt Basic,Olahraga,Rp75.000,5,Kartu Kredit,2026-09-09,Tangerang Selatan,1.0
3,TRX00121,"Ilsa Mahendra, S.T.",Quia Pro,Elektronik,50000.0,1,transfer bank,16-08-2026,Bengkulu,4.0
4,TRX00222,NaN,Magnam,FASHION,50000,2,Kartu Kredit,2026-06-28,NaN,2.0


## **K-3. Deteksi dan Penanganan Missing Value**

In [6]:
# Cek jumlah missing value per kolom
print(df.isnull().sum())

# Penanganan missing value
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Jumlah baris setelah drop_duplicates():", len(df))

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64
Jumlah baris setelah drop_duplicates(): 495


### **K-4. Deteksi dan Penanganan Duplicate**

In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


## **K-5. Koreksi Tipe Data dan Standardisasi Format**

In [8]:
# a. Standardisasi teks kategorikal (category, payment_method, shipping_city)
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# b. Koreksi tipe data pada kolom price (dari teks bercampur simbol menjadi numerik)
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

# c. Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

# d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

### **K-6. Ekspor Dataset Bersih**

In [9]:
path_bersih = os.path.join(DIR_SIMPAN, "transaksi_bersih_latihan.csv")
df.to_csv(path_bersih, index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


**Kesimpulannya,**
jumlah baris yang dihasilkan dengan SEED 7 adalah 515 baris untuk data mentah dan 490 baris untuk data bersih. Jumlah total baris akhir ini ternyata sama persis dengan hasil saat menggunakan SEED = 42.

**Kenapa sama?**

Pertama untuk Data Mentah yaitu 515 baris, jumlah ini bersifat statis dan tidak terpengaruh oleh random seed karena secara eksplisit ditulis dalam kode bahwa data utama dibangkitkan sebanyak 500 baris (N = 500), lalu disuntikkan secara paksa 15 baris duplikat (n=15), sehingga totalnya pasti 515 baris.
Lalu untuk Data Bersih sebanyak 490 tetap sama karena parameter penentu kerusakan data menggunakan proporsi yang bernilai tetap, yaitu frac untuk persentase nilai kosong dan pencabutan 5 baris duplikat unik saat drop_duplicates() dijalankan.

**Tetapi terdapat perbedaan antara SEED 42 vs SEED 7 yaitu :**

Pada SEED 42 jumlah nilai kosong pada kolom rating bernilai 166, tetapi pada SEED 7 turun menjadi 120. Ini terjadi karena nilainya dibangkitkan secara acak menggunakan perintah random.choice([1, 2, 3, 4, 5, None, None])(Pada Langkah K-2). Mengubah nilai SEED menjadi 7 akan merombak seluruh pola pengacakan pada sistem. Akibatnya, frekuensi terpilihnya nilai None ikut berubah dari 166 kali menjadi 120 kali, murni karena perbedaan urutan tarikan acak pada seed yang baru.


# **Latihan 2**- Menambahkan kolom **is_valid_price** bernilai True jika price > 0.
Digunakan untuk memeriksa apakah ada harga tidak valid.

In [10]:
# Tambahkan kolom is_valid_price pada dataframe df
df['is_valid_price'] = df['price'] > 0

# Cek hasil pemeriksaan harga
print(df['is_valid_price'].value_counts())

is_valid_price
True    490
Name: count, dtype: int64


Hasil dari penambahan kolom is_valid_price (dengan kondisi price > 0) menunjukkan nilai True sebanyak 490 baris. Tidak ditemukan satu pun transaksi bernilai False. Hal ini membuktikan bahwa seluruh 490 transaksi di dalam dataset bersih memiliki angka nominal harga yang valid (lebih dari 0 Rupiah).

Bisa dilihat pada tabel dibawah, kolom is_valid_price terbukti berhasil ditambahkan

In [11]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur Basic,Rumah Tangga,120000.0,2,Transfer Bank,2026-09-14,Bau-Bau,1.0,True
1,TRX00430,Faizah Kusumo,Voluptates Max,Olahraga,250000.0,3,Transfer Bank,2026-07-17,Cilegon,4.0,True
2,TRX00083,"Drs. Sari Aryani, M.TI.",Deserunt Basic,Olahraga,75000.0,5,Kartu Kredit,2026-09-09,Tangerang Selatan,1.0,True
3,TRX00121,"Ilsa Mahendra, S.T.",Quia Pro,Elektronik,500000.0,1,Transfer Bank,2026-08-16,Bengkulu,4.0,True
5,TRX00148,Cahyanto Agustina,Officia Basic,Buku,15000.0,5,Transfer Bank,2026-08-14,Tidore Kepulauan,4.0,True


## **Latihan 3** - Menghitung jumlah transaksi per category menggunakan **value_counts()** pada dataset yang sudah bersih.

In [12]:
# Hitung jumlah transaksi per category menggunakan value_counts()
jumlah_per_kategori = df['category'].value_counts()

print("Jumlah transaksi per kategori:")
print(jumlah_per_kategori)

Jumlah transaksi per kategori:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


Bisa dilihat pada gambar diatas eksekusi fungsi value_counts() pada kolom category di dataset bersih menghasilkan rincian sebaran penjualan sebagai berikut:

1.	Rumah Tangga: 91 transaksi
2.	Kesehatan: 86 transaksi
3.	Buku: 81 transaksi
4.	Fashion: 80 transaksi
5.	Elektronik: 78 transaksi
6.	Olahraga: 74 transaksi
